# Lead-Lag Scout

**Purpose:** Quickly browse related Kalshi markets, pull history, plot paths, and compute a simple lag metric.
This is **exploration** only

## 0. Setup

In [1]:
from __future__ import annotations

import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

# Resolve repo root whether the kernel cwd is repo/ or notebooks/
_candidates = [Path.cwd(), Path.cwd().parent, Path("..").resolve()]
ROOT = next(
    p
    for p in _candidates
    if (p / "pyproject.toml").exists() and (p / "src" / "signalgraph").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))

from signalgraph.ingestion.kalshi import KalshiClient
from signalgraph.normalization.resample import resample_observations
from signalgraph.normalization.schema import (
    NormalizedFrame,
    kalshi_candlesticks_to_observations,
)
from signalgraph.research.lead_lag import (
    lead_lag_correlation,
    run_multiple_horizons,
)
from signalgraph.research.returns import calculate_probability_change
from signalgraph.storage import save_raw_payload

RAW_DIR = ROOT / "data" / "raw"
ARCHIVE_RAW = True  # keep immutable API dumps under data/raw/

print("ROOT:", ROOT)

ROOT: /Users/jasoncharwin/Personal Code Projects/SignalGraph


## 1. Browse a series

Change `SERIES` to scout other families (`KXCPI`, `KXGDP`, `KXRATECUT`, …).

In [3]:
SERIES = "CONTROLS"
STATUS = "open"
LIMIT = 30

with KalshiClient() as client:
    payload = client.list_markets(limit=LIMIT, series_ticker=SERIES, status=STATUS)
    if ARCHIVE_RAW:
        path = save_raw_payload(payload, RAW_DIR, label=f"scout_{SERIES}")
        print("archived:", path)

markets = payload.data.get("markets", []) if isinstance(payload.data, dict) else []
browse = pl.DataFrame(
    [
        {
            "ticker": m.get("ticker"),
            "title": (m.get("title") or "")[:90],
            "event": m.get("event_ticker"),
            "yes_bid": m.get("yes_bid_dollars"),
            "yes_ask": m.get("yes_ask_dollars"),
            "last": m.get("last_price_dollars"),
            "volume": m.get("volume_fp"),
            "oi": m.get("open_interest_fp"),
            "close_time": m.get("close_time"),
        }
        for m in markets
    ]
)
browse

archived: /Users/jasoncharwin/Personal Code Projects/SignalGraph/data/raw/kalshi/markets/20260905T203908Z_scout_CONTROLS.json


ticker,title,event,yes_bid,yes_ask,last,volume,oi,close_time
str,str,str,str,str,str,str,str,str
"""CONTROLS-2028-R""","""Will the Republican party win …","""CONTROLS-2028""","""0.4300""","""0.5000""","""0.4600""","""1879.60""","""1111.60""","""2029-02-01T15:00:00Z"""
"""CONTROLS-2028-D""","""Will the Democratic party win …","""CONTROLS-2028""","""0.5100""","""0.5700""","""0.5700""","""2015.74""","""78.78""","""2029-02-01T15:00:00Z"""
"""CONTROLS-2026-R""","""Will Republicans win the U.S. …","""CONTROLS-2026""","""0.5200""","""0.5300""","""0.5300""","""4562136.89""","""2580853.26""","""2027-02-01T15:00:00Z"""
"""CONTROLS-2026-D""","""Will Democrats win the U.S. Se…","""CONTROLS-2026""","""0.4600""","""0.4700""","""0.4700""","""3442514.88""","""1717313.35""","""2027-02-01T15:00:00Z"""


## 2. Pick two related markets

Choose a **source** you think might lead and a **target** that might follow.

Default: adjacent Fed threshold contracts (same meeting ladder).

In [9]:
#Source is the market that might move first
#Target is the market that might follow

SOURCE = "SENATETX-26-D" #Texas Democrat hypothesized win
TARGET = "CONTROLS-2026-D" #Senate win hypothesized
TICKERS = [SOURCE, TARGET]

# Lookback window and candle grain (Kalshi allows 1, 60, 1440 minutes).
LOOKBACK_DAYS = 14 #how many days to pull back
PERIOD_INTERVAL = 60  # 1-hour candles; resample later if needed
GRID_MINUTES = 60  # research grid after forward-fill

end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=LOOKBACK_DAYS)
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

print(f"{SOURCE}  →  {TARGET}")
print(f"window: {start_dt.isoformat()} → {end_dt.isoformat()}")
print(f"candles={PERIOD_INTERVAL}m, grid={GRID_MINUTES}m")

SENATETX-26-D  →  CONTROLS-2026-D
window: 2026-08-22T21:18:31.575862+00:00 → 2026-09-05T21:18:31.575862+00:00
candles=60m, grid=60m


## 3. Fetch history + normalize

In [ ]:
frames: list[pl.DataFrame] = []
meta_rows: list[dict] = []

with KalshiClient() as client:
    for ticker in TICKERS:
        detail = client.get_market(ticker)
        market = detail.data.get("market", {}) if isinstance(detail.data, dict) else {}
        series_ticker = market.get("series_ticker")

        hist = client.get_history(
            ticker,
            series_ticker=series_ticker,
            start_ts=start_ts,
            end_ts=end_ts,
            period_interval=PERIOD_INTERVAL,
        )
        if ARCHIVE_RAW:
            save_raw_payload(hist, RAW_DIR, label=f"scout_hist_{ticker}")

        candles = (
            hist.data.get("candlesticks", [])
            if isinstance(hist.data, dict)
            else []
        )
        obs = kalshi_candlesticks_to_observations(
            candlesticks=candles,
            market_id=ticker,
            event_id=market.get("event_ticker"),
            market_title=market.get("title"),
        )
        frame = NormalizedFrame.from_observations(obs)
        frames.append(frame)

        meta_rows.append(
            {
                "ticker": ticker,
                "title": (market.get("title") or "")[:100],
                "series": series_ticker,
                "n_candles": len(candles),
                "yes_bid": market.get("yes_bid_dollars"),
                "yes_ask": market.get("yes_ask_dollars"),
                "volume": market.get("volume_fp"),
                "rules": (market.get("rules_primary") or market.get("subtitle") or "")[:120],
            }
        )
        print(f"{ticker}: {len(candles)} candles")

meta = pl.DataFrame(meta_rows)
raw = pl.concat(frames) if frames else NormalizedFrame.empty()
meta

## 4. Align on a time grid + probability changes

Forward-fill only (no look-ahead).

In [ ]:
if raw.is_empty():
    raise RuntimeError("No candlestick data returned — try another series/window.")

aligned = resample_observations(raw, interval_minutes=GRID_MINUTES)
aligned = calculate_probability_change(aligned, price_col="yes_mid")

summary = (
    aligned.group_by("market_id")
    .agg(
        pl.len().alias("n"),
        pl.col("timestamp").min().alias("start"),
        pl.col("timestamp").max().alias("end"),
        pl.col("yes_mid").null_count().alias("mid_nulls"),
        (pl.col("yes_ask") - pl.col("yes_bid")).mean().alias("avg_spread"),
        (pl.col("prob_change") == 0).mean().alias("frac_unchanged"),
    )
    .sort("market_id")
)
summary

## 5. Plot probability paths

Ask: does one market jump while the other stays flat, then catch up?

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

for ticker in TICKERS:
    sub = aligned.filter(pl.col("market_id") == ticker).sort("timestamp")
    axes[0].plot(
        sub["timestamp"].to_list(),
        sub["yes_mid"].to_list(),
        label=ticker,
        linewidth=1.5,
    )
    axes[1].plot(
        sub["timestamp"].to_list(),
        sub["prob_change"].to_list(),
        label=ticker,
        linewidth=1.0,
        alpha=0.85,
    )

axes[0].set_ylabel("yes_mid")
axes[0].set_title("Exploratory: implied probability paths")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel(r"$\Delta p$")
axes[1].set_xlabel("UTC timestamp")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Quick lead-lag metrics

Correlation of $\Delta p_{source,t}$ with $\Delta p_{target,t+k}$ for several horizons.

Also try the reverse direction — if both look similar, you may just be seeing co-movement.

In [ ]:
HORIZONS = [0, 1, 2, 3, 6, 12]  # steps on GRID_MINUTES (e.g. 1 ≈ 60m)

def lag_table(source: str, target: str) -> pl.DataFrame:
    corr_rows = []
    for h in HORIZONS:
        corr_rows.append(
            {
                "source": source,
                "target": target,
                "lag_steps": h,
                "lag_minutes": h * GRID_MINUTES,
                "corr": lead_lag_correlation(
                    aligned,
                    source_market=source,
                    target_market=target,
                    lag=h,
                ),
            }
        )
    return pl.DataFrame(corr_rows)

forward = lag_table(SOURCE, TARGET)
reverse = lag_table(TARGET, SOURCE)

print("Forward (source leads target?)")
display(forward)
print("Reverse")
display(reverse)

In [ ]:
# OLS across horizons (exploratory only)
reg = run_multiple_horizons(
    aligned,
    source_market=SOURCE,
    target_market=TARGET,
    horizons=[h for h in HORIZONS if h > 0],
)
reg.select(
    [
        "source_market",
        "target_market",
        "lag",
        "coefficient",
        "t_stat",
        "p_value",
        "r_squared",
        "observations",
        "correlation",
    ]
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(forward["lag_minutes"].to_list(), forward["corr"].to_list(), marker="o", label=f"{SOURCE} → {TARGET}")
ax.plot(reverse["lag_minutes"].to_list(), reverse["corr"].to_list(), marker="o", label=f"{TARGET} → {SOURCE}")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("lag (minutes)")
ax.set_ylabel("corr(Δp_source_t, Δp_target_t+k)")
ax.set_title("Exploratory lead-lag correlation curve")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Scratch observations

Fill this in before writing a formal hypothesis.

- Visual lead-lag episodes?
- Stronger forward or reverse?
- Wide spreads / many unchanged intervals?
- Worth expanding to a 5–15 market universe?

**Next:** if something looks interesting, document the economic story in `research/hypotheses.md`, then populate `config/market_groups.yaml` → `research_universe_v1`.